# Лабораторная работа №1 — очистка данных

Подготовка исходного CSV с использованием Apache Spark и сохранение очищенных данных в Apache Iceberg.

Текущий набор данных используется как учебный для проверки пайплайна.


## 1. Инициализация Spark

Используется общая конфигурация проекта из `src/spark_session.py`.


In [ ]:
from src.spark_session import create_spark

spark = create_spark("Lab1DataCleansing")
spark


## 2. Параметры

Для перехода на другой датасет основные пути и имя таблицы меняются здесь.


In [ ]:
INPUT_PATH = "/app/data/raw/used_cars_data.csv"
TABLE_NAME = "local.lab1.used_cars"

# None — использовать весь датасет.
# Для тестового запуска можно указать, например, 100_000.
ROW_LIMIT = None

print("CSV:", INPUT_PATH)
print("Iceberg:", TABLE_NAME)
print("Limit:", ROW_LIMIT)


## 3. Чтение исходного CSV

На первом этапе автоматическое определение типов отключено.  
Все поля читаются как строки, а нужные типы задаются явно.


In [ ]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .csv(INPUT_PATH)
)

if ROW_LIMIT is not None:
    df_raw = df_raw.limit(ROW_LIMIT)

print("Количество столбцов:", len(df_raw.columns))
print("Количество партиций:", df_raw.rdd.getNumPartitions())

df_raw.printSchema()


## 4. Просмотр исходных данных


In [ ]:
df_raw.show(10, truncate=False)


## 5. Выбор признаков

Для учебного автомобильного датасета оставляем 15 признаков, используемых в дальнейшей обработке.


In [ ]:
SELECTED_COLUMNS = [
    "vin",
    "body_type",
    "daysonmarket",
    "fleet",
    "has_accidents",
    "horsepower",
    "is_certified",
    "is_cpo",
    "is_oemcpo",
    "major_options",
    "maximum_seating",
    "mileage",
    "price",
    "wheel_system",
    "year",
]

df = df_raw.select(*SELECTED_COLUMNS)

df.show(10, truncate=False)


## 6. Приведение типов и преобразование признаков


In [ ]:
from pyspark.sql.functions import (
    col,
    lit,
    regexp_extract_all,
    regexp_replace,
)

df = (
    df
    .withColumn(
        "daysonmarket",
        col("daysonmarket").cast("integer"),
    )
    .withColumn(
        "fleet",
        col("fleet").cast("boolean"),
    )
    .withColumn(
        "has_accidents",
        col("has_accidents").cast("boolean"),
    )
    .withColumn(
        "horsepower",
        col("horsepower").cast("float"),
    )
    .withColumn(
        "is_certified",
        col("is_certified").cast("boolean"),
    )
    .withColumn(
        "is_cpo",
        col("is_cpo").cast("boolean"),
    )
    .withColumn(
        "is_oemcpo",
        col("is_oemcpo").cast("boolean"),
    )
    .withColumn(
        "major_options",
        regexp_extract_all(
            col("major_options"),
            lit(r"'([^']*)'"),
            1,
        ),
    )
    .withColumn(
        "maximum_seating",
        regexp_replace(
            col("maximum_seating"),
            r"\s+seats",
            "",
        ).cast("integer"),
    )
    .withColumn(
        "mileage",
        col("mileage").cast("float"),
    )
    .withColumn(
        "price",
        col("price").cast("float"),
    )
    .withColumn(
        "year",
        col("year").cast("integer"),
    )
)

df.printSchema()


## 7. Проверка результата


In [ ]:
df.show(10, truncate=False)


## 8. Сохранение в Apache Iceberg

Таблица создаётся заново либо заменяется при повторном запуске notebook.


In [ ]:
spark.sql(
    "CREATE NAMESPACE IF NOT EXISTS local.lab1"
)

(
    df.writeTo(TABLE_NAME)
    .using("iceberg")
    .createOrReplace()
)

print("Таблица успешно сохранена:", TABLE_NAME)


## 9. Проверка сохранённой Iceberg-таблицы


In [ ]:
df_saved = spark.table(TABLE_NAME)

print("Количество строк:", f"{df_saved.count():,}")

df_saved.printSchema()
df_saved.show(10, truncate=False)


## 10. Проверка каталога Iceberg


In [ ]:
spark.sql(
    "SHOW TABLES IN local.lab1"
).show(truncate=False)
